# Week 5 Lab Solution: Algorithm Analysis and Search Implementations
This notebook provides solutions and explanations for all exercises in the Week 5 Lab.

## 1. Setup and Data Preparation

In [ ]:
import random
import time
import matplotlib.pyplot as plt

# Utility: Generate random dataset
def generate_dataset(size, sorted_data=False):
    data = random.sample(range(size * 10), size)
    return sorted(data) if sorted_data else data

# Dataset sizes to test
sizes = [10**3, 10**4, 10**5, 10**6]

## 2. Implement Linear and Binary Search

In [ ]:
def linear_search(arr, key):
    for i, val in enumerate(arr):
        if val == key:
            return i
    return -1

def binary_search(arr, key):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == key:
            return mid
        elif arr[mid] < key:
            low = mid + 1
        else:
            high = mid - 1
    return -1

## 3. Benchmarking and Visualization
Compare the performance of Linear Search and Binary Search across various dataset sizes.

In [ ]:
def benchmark_search(search_fn, arr, key):
    start = time.time()
    search_fn(arr, key)
    return time.time() - start

results = {"size": [], "linear": [], "binary": []}

for size in sizes:
    data_unsorted = generate_dataset(size, sorted_data=False)
    data_sorted = sorted(data_unsorted)
    key = data_unsorted[-1]  # Worst-case key

    t_linear = benchmark_search(linear_search, data_unsorted, key)
    t_binary = benchmark_search(binary_search, data_sorted, key)

    results["size"].append(size)
    results["linear"].append(t_linear)
    results["binary"].append(t_binary)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(results["size"], results["linear"], label='Linear Search', marker='o')
plt.plot(results["size"], results["binary"], label='Binary Search', marker='s')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Input Size (log scale)")
plt.ylabel("Time Taken (seconds, log scale)")
plt.title("Search Algorithm Benchmark")
plt.legend()
plt.grid(True)
plt.show()

## 4. Complex and Practical Exercises
### 4.1: Search on File-based Datasets

In [ ]:
def search_file_dataset(file_path, key):
    with open(file_path, 'r') as f:
        arr = [int(line.strip()) for line in f if line.strip()]

    # Linear search
    start = time.time()
    idx_linear = linear_search(arr, key)
    t_linear = time.time() - start

    # Binary search (requires sorted array)
    arr_sorted = sorted(arr)
    start = time.time()
    idx_binary = binary_search(arr_sorted, key)
    t_binary = time.time() - start

    print(f"Linear Search: Index={idx_linear}, Time={t_linear:.6f}s")
    print(f"Binary Search: Index={idx_binary}, Time={t_binary:.6f}s")
    return idx_linear, t_linear, idx_binary, t_binary

### 4.2: Adaptive Hybrid Search

In [ ]:
def hybrid_search(arr, key, threshold=1000):
    if len(arr) < threshold:
        return linear_search(arr, key)
    is_sorted = all(arr[i] <= arr[i+1] for i in range(len(arr)-1))
    if is_sorted:
        return binary_search(arr, key)
    else:
        return binary_search(sorted(arr), key)

# Benchmark hybrid search
results["hybrid"] = []
for size in sizes:
    data_unsorted = generate_dataset(size, sorted_data=False)
    key = data_unsorted[-1]
    t_hybrid = benchmark_search(hybrid_search, data_unsorted, key)
    results["hybrid"].append(t_hybrid)

# Plot all three
plt.figure(figsize=(10, 6))
plt.plot(results["size"], results["linear"], label='Linear Search', marker='o')
plt.plot(results["size"], results["binary"], label='Binary Search', marker='s')
plt.plot(results["size"], results["hybrid"], label='Hybrid Search', marker='^')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Input Size (log scale)")
plt.ylabel("Time Taken (seconds, log scale)")
plt.title("Search Algorithm Benchmark (with Hybrid)")
plt.legend()
plt.grid(True)
plt.show()

## 5. Summary and Reflections
- **Trade-offs:** Linear search is simple and works on unsorted data but is slow for large datasets (O(n)). Binary search is much faster (O(log n)) but requires sorted data.
- **Runtime vs Theory:** In practice, binary search is orders of magnitude faster for large datasets, but for very small arrays, linear search may be competitive due to lower overhead.
- **When Linear Search?** For small arrays, unsorted data, or when data is nearly always found at the start.
- **Optimal Search for Large Datasets:** Use distributed indexes, partitioned data, and parallel search algorithms. Consider data locality and network latency.

## Optional Challenge: Interpolation Search

In [ ]:
def interpolation_search(arr, key):
    low = 0
    high = len(arr) - 1
    while low <= high and arr[low] <= key <= arr[high]:
        if arr[high] == arr[low]:
            if arr[low] == key:
                return low
            else:
                return -1
        pos = low + ((key - arr[low]) * (high - low) // (arr[high] - arr[low]))
        if pos < 0 or pos >= len(arr):
            return -1
        if arr[pos] == key:
            return pos
        elif arr[pos] < key:
            low = pos + 1
        else:
            high = pos - 1
    return -1

# Test and compare interpolation search
size = 10**5
arr = sorted(random.sample(range(size * 10), size))
key = arr[-1]
t_binary = benchmark_search(binary_search, arr, key)
t_interp = benchmark_search(interpolation_search, arr, key)
print(f"Binary Search: {t_binary:.6f}s, Interpolation Search: {t_interp:.6f}s")